# 3. Dépendances et groupes C-RDG

**Question.** Reconstruit-on des liens pertinents, des sous-groupes valides ou des groupes complets ?

Ces tâches ont des dénominateurs différents. Une vue locale à un saut peut être correcte et incomplète. L'absence de groupe établi localement n'affirme pas son absence globale. Le graphe consomme les diagnostics propres et réellement reçus ; l'oracle reste un évaluateur hors ligne.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from cav_recovery.laboratory.workbench import Laboratory
from cav_recovery.laboratory.storage import json_file

lab = Laboratory.from_environment()
display(HTML(lab.table_html([{'résultats': str(lab.root), 'M02': str(lab.dataset)}])))


## Liens et groupes : séparer validité et complétude

B est ici le comptage simple de demandeurs confirmés. C ajoute leur cohérence spatiale. Les métriques de groupes exacts sanctionnent un sous-groupe correct mais incomplet ; la validité partielle répond à une autre question.


In [ ]:
mesures = lab.historical_metrics()
display(HTML(lab.table_html([m for m in mesures if m['task'] in ('links_global','links_observable','B_groups','C_groups')])))
fig = lab.plot_historical_graphs()
plt.show()


## Un groupe collectif peut être acyclique

Dans `collective-1`, graine 21019, à 56,4 s, la vue de V1 contient V3 → ressource → V1 → ressource → V4. Deux demandeurs sont contraints, trois CAV sont liés et aucun cycle n'est nécessaire. Les ressources sont des espaces requis par les mouvements, pas une seconde classe de véhicules.

La preuve est lue ci-dessous dans le résultat indépendant enregistré ; aucune arête n'est inventée pour le dessin.


In [ ]:
collectif = next(r for r in lab.historical_episodes() if r['name']=='collective-1')
display(HTML(lab.table_html(collectif['acyclic_collective_appointments'])))
audit = json_file(lab.historical / 'final-audits' / 'additional-metrics.json')
display(HTML(lab.table_html([{'référence': k, **v} for k,v in audit['indirect_metrics'].items()])))
from IPython.display import Image
capture = lab.root / 'evidence' / 'interface_0512af2' / 'graph.png'
if capture.exists():
    display(Image(filename=str(capture)))
    print('Vue acceptée à 60 s, collective-1, graine 21019, commit 0512af2. '
          'La première preuve acyclique ci-dessus est à 56,4 s.')


## Apport au TCN : une ablation explicite

La variante TCN brute n'a aucun canal D01/C-RDG. La variante D01 ajoute quatre statuts locaux. La variante D01+C-RDG ajoute le nombre de relations actuelles et de groupes observés localement. Ce sont des connaissances accessibles au CAV à cet instant, pas les liens de référence globaux.


In [ ]:
from cav_recovery.laboratory.features import BASE_FEATURES, DIAGNOSTIC_FEATURES, GRAPH_FEATURES
display(HTML(lab.table_html([
    {'variante': 'TCN brut', 'canaux': len(BASE_FEATURES), 'extension': 'aucune'},
    {'variante': 'TCN + D01', 'canaux': len(BASE_FEATURES)+4, 'extension': ', '.join(DIAGNOSTIC_FEATURES)},
    {'variante': 'TCN + D01 + C-RDG', 'canaux': len(BASE_FEATURES)+6, 'extension': ', '.join(GRAPH_FEATURES)},
])))
display(HTML(lab.table_html(lab.model_results('integrated'))))


**Interprétation.** Un gain éventuel du modèle temporel grâce à ces deux compteurs ne démontre pas une reconstruction complète du graphe ni une racine causale dynamique unique. Les scores historiques exacts et le faible rappel indirect restent conservés.
